# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ibraheem-ali23/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

Lane: Content Refresh & Discoverability.
ML Task Type: Supervised Binary Classification with Probability Scoring/Ranking. We estimate the probability that an existing page is experiencing organic traffic decline, then rank pages by that predicted risk score so content editors review and refresh the highest-priority pages first.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess
import pandas as pd
import numpy as np

if "google.colab" in sys.modules and not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ibraheem-ali23/flyrank-ml-internship.git"], check=True)
    os.chdir("flyrank-ml-internship")
elif not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.path.exists("../../data/raw/content_refresh_anonymized.csv"):
    os.chdir("../..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Loaded starter dataset slice: {df.shape[0]:,} rows, {df.shape[1]} columns")


Loaded starter dataset slice: 30,000 rows, 44 columns


## 2. Target or proxy

Target Proxy: `is_declining_label` (1 when `trend_direction == 'down'`, and 0 otherwise). This label comes from an observed 90-day traffic trend bucket in the starter dataset, acting as a practical proxy for content decay. Direct outcome calculations like `trend_pct` are excluded from model features to avoid data leakage.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print("Target proxy distribution (is_declining_label):")
print(df["is_declining_label"].value_counts(normalize=True).round(3))

Target proxy distribution (is_declining_label):
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64


## 3. Success metric

Success Metric: Precision@K (specifically Precision@20 and Precision@50) evaluated on a client-holdout split. Since editorial teams have limited weekly capacity to refresh K pages, Precision@K measures the fraction of top-K flagged pages that are genuinely declining. A good model should beat the baseline declining rate (~54%) and outperform the simple hand rule at K=50 (targeting >= 0.65 Precision@50).

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

baseline_rate = df["is_declining_label"].mean()
print(f"Random baseline Precision@50: {baseline_rate:.3f}")

Random baseline Precision@50: 0.542


## 4. The unit of analysis, as a real dataframe

Unit of Analysis: One row represents one anonymized client web page measured over a 90-day window, pairing pre-decision observable signals (age, update recency, impressions, rank position, CTR, word count) with our binary target column (`is_declining_label`).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
unit_df = df[features + ["is_declining_label"]].copy()
print(f"Unit of analysis table (1 row = 1 page): {unit_df.shape[0]:,} pages")
unit_df.head(10)


Unit of analysis table (1 row = 1 page): 30,000 pages


,content_age_days,days_since_last_update,impressions_90d,avg_position,ctr,word_count,is_declining_label
0,187,20,3803,10.6,0.76,3221.0,1
1,445,25,15320,20.3,0.05,2481.0,1
2,141,20,12581,36.5,0.09,3515.0,1
3,463,22,11751,6.2,0.49,NaN,0
4,263,14,19140,44.0,0.13,2803.0,1
5,147,20,3970,8.5,0.03,3080.0,1
6,90,20,20,7.0,0.00,3059.0,1
7,445,22,1724,21.2,0.06,NaN,0
8,90,20,32574,46.0,0.09,3807.0,1
9,257,104,1240,4.9,0.16,NaN,1


## 5. Why ML beats a fixed rule here

Why ML Beats a Fixed Rule: Fixed rules (like flagging pages with `days_since_last_update >= 180` and `impressions_90d >= 500`) rely on rigid cutoffs and create large blocks of tied scores. Real content decay depends on non-linear interactions across rank position, CTR, page age, and impressions. An ML model captures these multi-signal interactions to support the concrete action of prioritizing high-impact page refreshes.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.tree import DecisionTreeClassifier

stale = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
hand_rule_score = stale * visible * df["impressions_90d"]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values
tree = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42).fit(X, y)
tree_score = tree.predict_proba(X)[:, 1]

print(f"Hand rule Precision@50: {precision_at_k(hand_rule_score, y, 50):.3f}")
print(f"Depth-3 ML Tree Precision@50: {precision_at_k(tree_score, y, 50):.3f}")

Hand rule Precision@50: 0.680
Depth-3 ML Tree Precision@50: 0.720


italicized text## Self-check

Before you submit, confirm each line honestly:

- [x ] Every section above is filled — markdown thinking AND the code that backs it
- [x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x] No client names, URLs, or private queries anywhere
- [x ] My claims use careful words: observed, measured, directional, decision-support
- [ x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.